In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "ml_df_v4.csv").exists() and (ROOT.parent / "data" / "ml_df_v4.csv").exists():
    ROOT = ROOT.parent

ml_df = pd.read_csv(ROOT / "data" / "ml_df_v4.csv")

LEAKAGE_COLS = {
    "home_goals", "away_goals", "score", "result",
    "home_team_win", "away_team_win", "draw",
    "extra_time", "penalty_shootout", "score_penalties",
}

CATEGORICAL_COLS = ["stage", "tournament_name", "host_country", "home_team", "away_team", "tournament_size_category"]
ENGINEERED_PREFIXES = ["home_", "away_"]
EXPLICIT_COLS = [
    "win_rate_diff", "goal_diff_diff", "form_diff", "goals_per_match_diff", "conceded_per_match_diff",
    "season_win_rate_diff", "season_goal_diff_diff", "elo_diff", "ranking_diff", "wc_goals_diff_before",
    "total_teams", "matches_played", "goals_scored_tournament", "avg_goals_per_game", "year_normalized",
    "is_neutral_venue", "home_advantage_strength",
]

feature_cols = [
    c for c in ml_df.columns
    if c in CATEGORICAL_COLS or any(c.startswith(p) for p in ENGINEERED_PREFIXES) or c in EXPLICIT_COLS
]
feature_cols = [c for c in feature_cols if c not in LEAKAGE_COLS and c != "result_target"]

In [2]:
# Classify each feature: Safe (uses shift/before logic), External (ranking, elo), Suspicious
safe_patterns = ["_before", "_diff", "_rate", "_per_match", "_season", "total_teams", "matches_played",
                 "goals_scored_tournament", "avg_goals_per_game", "year_normalized", "stage", "tournament",
                 "host_country", "home_team", "away_team", "tournament_size"]
external = ["ranking_score", "elo_before"]

classified = []
for c in sorted(feature_cols):
    if any(ex in c for ex in ["ranking_score", "elo_before"]):
        cat = "External"
    elif any(p in c for p in safe_patterns) or c in CATEGORICAL_COLS or c in EXPLICIT_COLS:
        cat = "Safe"
    elif "_after" in c or "_post" in c:
        cat = "Suspicious"
    else:
        cat = "Review"
    classified.append((c, cat))

df_audit = pd.DataFrame(classified, columns=["feature", "classification"])
print("Classification summary:")
print(df_audit["classification"].value_counts())
print()
if (df_audit["classification"] == "Suspicious").any():
    print("SUSPICIOUS:", df_audit[df_audit["classification"] == "Suspicious"]["feature"].tolist())
if (df_audit["classification"] == "Review").any():
    print("REVIEW:", df_audit[df_audit["classification"] == "Review"]["feature"].tolist())

Classification summary:
classification
Safe        150
Review        6
External      4
Name: count, dtype: int64

REVIEW: ['away_is_defending_champion_x', 'away_is_defending_champion_y', 'away_last5_points', 'home_is_defending_champion_x', 'home_is_defending_champion_y', 'home_last5_points']
